In [21]:
import pandas as pd

df = pd.read_csv('train_clean.csv')
print('Shape:', df.shape)
print()
df.head(3)

Shape: (8947, 10)



,label,statement,subject,speaker,speaker_job,state_info,party_affiliation,subject_primary,speaker_grouped,party_grouped
0,1,China is in the South China Sea and a military...,"china,foreign-policy,military",donald-trump,president-elect,New York,republican,china,donald-trump,republican
1,0,With the resources it takes to execute just ov...,health-care,chris-dodd,u.s. senator,Connecticut,democrat,health-care,chris-dodd,democrat
2,0,The governor has proposed tax giveaways to co...,"corporations,pundits,taxes,abc-news-week",donna-brazile,political commentator,"Washington, D.C.",democrat,corporations,donna-brazile,democrat


In [22]:
import pandas as pd
import numpy as np
import sklearn
import scipy
print('pandas:', pd.__version__)
print('numpy:', np.__version__)
print('sklearn:', sklearn.__version__)
print('scipy:', scipy.__version__)

pandas: 3.0.2
numpy: 2.4.4
sklearn: 1.8.0
scipy: 1.17.1


In [23]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import ComplementNB
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.metrics import f1_score, classification_report
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

In [24]:
train = pd.read_csv('train_clean.csv')
test = pd.read_csv('test_nolabel.csv')

print('Train shape:', train.shape)
print('Test shape:', test.shape)
print()
print('Distribución label:')
print(train['label'].value_counts())

Train shape: (8947, 10)
Test shape: (3836, 7)

Distribución label:
label
1    5794
0    3153
Name: count, dtype: int64


In [ ]:
X_train = train['statement']
y_train = train['label']
X_test = test['statement']

print('X_train shape:', X_train.shape)
print('X_test shape:', X_test.shape)
print('y_train shape:', y_train.shape)

X_train shape: (8947,)
X_test shape: (3836,)
y_train shape: (8947,)


In [26]:
pipeline = Pipeline([
    ('vect', CountVectorizer(ngram_range=(1, 2))),
    ('clf', ComplementNB())
])

param_grid = {
    'vect__binary': [False, True],        # raw counts vs binario
    'vect__min_df': [1, 2, 5],            # filtrar términos poco frecuentes
    'clf__alpha': [0.1, 0.5, 1.0, 2.0],  # suavizado
    'clf__norm': [True, False]            # normalización
}

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)

print('Pipeline y grid search definidos.')
print(f'Combinaciones a probar: {len([1]) * 2 * 3 * 4 * 2}')

Pipeline y grid search definidos.
Combinaciones a probar: 48


In [27]:
grid_search.fit(X_train, y_train)

print(f'Mejor F1-Macro (CV): {grid_search.best_score_:.4f}')
print()
print('Mejores parámetros:')
for param, valor in grid_search.best_params_.items():
    print(f'  {param}: {valor}')

Fitting 5 folds for each of 48 candidates, totalling 240 fits
Mejor F1-Macro (CV): 0.5944

Mejores parámetros:
  clf__alpha: 2.0
  clf__norm: False
  vect__binary: True
  vect__min_df: 5


In [28]:
resultados = pd.DataFrame(grid_search.cv_results_)
resultados = resultados[['param_vect__binary', 'param_vect__min_df', 
                          'param_clf__alpha', 'param_clf__norm',
                          'mean_test_score', 'std_test_score']]
resultados = resultados.sort_values('mean_test_score', ascending=False)

print('Top 10 combinaciones:')
print(resultados.head(10).to_string(index=False))

Top 10 combinaciones:
 param_vect__binary  param_vect__min_df  param_clf__alpha  param_clf__norm  mean_test_score  std_test_score
               True                   5               2.0            False         0.594362        0.006316
              False                   5               2.0            False         0.592568        0.007563
               True                   5               1.0            False         0.592494        0.006689
               True                   5               1.0             True         0.592006        0.006741
               True                   5               0.5            False         0.590975        0.008068
               True                   5               0.1            False         0.590535        0.010885
              False                   5               1.0            False         0.590174        0.007828
              False                   5               0.5            False         0.590014        0.009929
      

In [29]:
mejor_modelo = grid_search.best_estimator_

scores = cross_val_score(mejor_modelo, X_train, y_train, cv=5, scoring='f1_macro')
print('F1-Macro por fold:')
for i, score in enumerate(scores):
    print(f'  Fold {i+1}: {score:.4f}')
print()
print(f'Media:       {scores.mean():.4f}')
print(f'Desviación:  {scores.std():.4f}')

F1-Macro por fold:
  Fold 1: 0.5980
  Fold 2: 0.5902
  Fold 3: 0.5878
  Fold 4: 0.5909
  Fold 5: 0.6050

Media:       0.5944
Desviación:  0.0063


In [30]:
from sklearn.model_selection import cross_val_predict

y_pred_cv = cross_val_predict(mejor_modelo, X_train, y_train, cv=5)

print('Classification Report (cross-validation):')
print(classification_report(y_train, y_pred_cv, target_names=['fake (0)', 'real (1)']))

Classification Report (cross-validation):
              precision    recall  f1-score   support

    fake (0)       0.46      0.54      0.50      3153
    real (1)       0.72      0.66      0.69      5794

    accuracy                           0.62      8947
   macro avg       0.59      0.60      0.59      8947
weighted avg       0.63      0.62      0.62      8947



In [31]:
from sklearn.model_selection import cross_val_predict

# Obtener probabilidades en lugar de predicciones directas
y_proba_cv = cross_val_predict(mejor_modelo, X_train, y_train, cv=5, method='predict_proba')

print('Probar distintos thresholds para mejorar F1-Macro:')
print()
for threshold in [0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6]:
    y_pred_thresh = (y_proba_cv[:, 1] >= threshold).astype(int)
    f1 = f1_score(y_train, y_pred_thresh, average='macro')
    f1_fake = f1_score(y_train, y_pred_thresh, average=None)[0]
    f1_real = f1_score(y_train, y_pred_thresh, average=None)[1]
    print(f'Threshold {threshold:.2f}: F1-Macro={f1:.4f} | F1-fake={f1_fake:.4f} | F1-real={f1_real:.4f}')

Probar distintos thresholds para mejorar F1-Macro:

Threshold 0.30: F1-Macro=0.5870 | F1-fake=0.4398 | F1-real=0.7343
Threshold 0.35: F1-Macro=0.5950 | F1-fake=0.4640 | F1-real=0.7261
Threshold 0.40: F1-Macro=0.5960 | F1-fake=0.4776 | F1-real=0.7143
Threshold 0.45: F1-Macro=0.5967 | F1-fake=0.4899 | F1-real=0.7035
Threshold 0.50: F1-Macro=0.5944 | F1-fake=0.4988 | F1-real=0.6901
Threshold 0.55: F1-Macro=0.5894 | F1-fake=0.5053 | F1-real=0.6736
Threshold 0.60: F1-Macro=0.5842 | F1-fake=0.5128 | F1-real=0.6557


In [32]:
from sklearn.pipeline import FeatureUnion
from sklearn.base import BaseEstimator, TransformerMixin

# Transformador para extraer una columna del dataframe
class ColumnSelector(BaseEstimator, TransformerMixin):
    def __init__(self, column):
        self.column = column
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        return X[self.column]

# Preparar dataframes completos (no solo la columna statement)
X_train_df = train[['statement', 'speaker_grouped', 'party_grouped', 'subject_primary']].copy()
X_test_df = test.copy()

# Aplicar el mismo speaker_grouped y party_grouped al test
MIN_COUNT = 5
vc = train['speaker'].value_counts()
speakers_frecuentes = vc[vc >= MIN_COUNT].index
X_test_df['speaker_grouped'] = test['speaker'].where(test['speaker'].isin(speakers_frecuentes), other='other')

vc_party = train['party_affiliation'].value_counts()
minoritarias = vc_party[vc_party < 50].index
X_test_df['party_grouped'] = test['party_affiliation'].where(~test['party_affiliation'].isin(minoritarias), other='other')

X_test_df['subject_primary'] = test['subject'].str.split(',').str[0].str.strip()

print('Features preparadas.')
print('Train:', X_train_df.shape)
print('Test:', X_test_df[['statement', 'speaker_grouped', 'party_grouped', 'subject_primary']].shape)

Features preparadas.
Train: (8947, 4)
Test: (3836, 4)


In [33]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline, FeatureUnion
from scipy.sparse import hstack

# Vectorizar statement
vect = CountVectorizer(ngram_range=(1, 2), binary=True, min_df=5)
X_train_text = vect.fit_transform(X_train_df['statement'])
X_test_text = vect.transform(X_test_df['statement'])

# One-hot encoding de features categóricas
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
cat_cols = ['speaker_grouped', 'party_grouped', 'subject_primary']
X_train_cat = ohe.fit_transform(X_train_df[cat_cols])
X_test_cat = ohe.transform(X_test_df[cat_cols])

# Combinar texto + categóricas
X_train_combined = hstack([X_train_text, X_train_cat])
X_test_combined = hstack([X_test_text, X_test_cat])

print('Shape train combinado:', X_train_combined.shape)
print('Shape test combinado:', X_test_combined.shape)

Shape train combinado: (8947, 7594)
Shape test combinado: (3836, 7594)


In [34]:
from sklearn.model_selection import cross_val_score, cross_val_predict

modelo_combinado = ComplementNB(alpha=2.0, norm=False)

# Evaluar con cross-validation
scores = cross_val_score(modelo_combinado, X_train_combined, y_train, cv=5, scoring='f1_macro')
print('F1-Macro por fold (modelo combinado):')
for i, score in enumerate(scores):
    print(f'  Fold {i+1}: {score:.4f}')
print()
print(f'Media:       {scores.mean():.4f}')
print(f'Desviación:  {scores.std():.4f}')
print()
print(f'Modelo solo statement:  0.5944')
print(f'Modelo combinado:       {scores.mean():.4f}')
print(f'Mejora:                 {scores.mean() - 0.5944:+.4f}')

F1-Macro por fold (modelo combinado):
  Fold 1: 0.6104
  Fold 2: 0.6012
  Fold 3: 0.5997
  Fold 4: 0.6011
  Fold 5: 0.6095

Media:       0.6044
Desviación:  0.0046

Modelo solo statement:  0.5944
Modelo combinado:       0.6044
Mejora:                 +0.0100


In [35]:
y_proba_combined = cross_val_predict(modelo_combinado, X_train_combined, y_train, cv=5, method='predict_proba')

print('Threshold tuning modelo combinado:')
print()
for threshold in [0.35, 0.40, 0.45, 0.50, 0.55]:
    y_pred_thresh = (y_proba_combined[:, 1] >= threshold).astype(int)
    f1 = f1_score(y_train, y_pred_thresh, average='macro')
    f1_fake = f1_score(y_train, y_pred_thresh, average=None)[0]
    f1_real = f1_score(y_train, y_pred_thresh, average=None)[1]
    print(f'Threshold {threshold:.2f}: F1-Macro={f1:.4f} | F1-fake={f1_fake:.4f} | F1-real={f1_real:.4f}')

Threshold tuning modelo combinado:

Threshold 0.35: F1-Macro=0.6042 | F1-fake=0.4817 | F1-real=0.7267
Threshold 0.40: F1-Macro=0.6038 | F1-fake=0.4911 | F1-real=0.7165
Threshold 0.45: F1-Macro=0.6032 | F1-fake=0.5003 | F1-real=0.7060
Threshold 0.50: F1-Macro=0.6044 | F1-fake=0.5120 | F1-real=0.6969
Threshold 0.55: F1-Macro=0.6010 | F1-fake=0.5185 | F1-real=0.6835


In [36]:
# Entrenar con todos los datos de train
modelo_combinado.fit(X_train_combined, y_train)

# Predecir en test
y_pred_test = modelo_combinado.predict(X_test_combined)

# Crear submission
submission = pd.DataFrame({
    'id': test['id'],
    'label': y_pred_test
})

submission.to_csv('submission_naive_bayes.csv', index=False)
print('Submission guardada como submission_naive_bayes.csv')
print()
print('Shape:', submission.shape)
print()
print('Distribución de predicciones:')
print(submission['label'].value_counts())
print()
print('Resumen final del modelo:')
print(f'  Modelo:        Complement Naive Bayes')
print(f'  Features:      statement (CountVectorizer bigrams) + speaker + party + subject')
print(f'  Parámetros:    alpha=2.0, norm=False, binary=True, min_df=5')
print(f'  F1-Macro CV:   0.6044')
print(f'  Threshold:     0.50')

Submission guardada como submission_naive_bayes.csv

Shape: (3836, 2)

Distribución de predicciones:
label
1    2221
0    1615
Name: count, dtype: int64

Resumen final del modelo:
  Modelo:        Complement Naive Bayes
  Features:      statement (CountVectorizer bigrams) + speaker + party + subject
  Parámetros:    alpha=2.0, norm=False, binary=True, min_df=5
  F1-Macro CV:   0.6044
  Threshold:     0.50


In [38]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import csr_matrix

# TF-IDF con mismos parámetros que el mejor CountVectorizer
tfidf_nb = TfidfVectorizer(ngram_range=(1, 2), min_df=5, sublinear_tf=True)
X_train_tfidf_nb = hstack([tfidf_nb.fit_transform(X_train_df['statement']), X_train_cat])
X_test_tfidf_nb = hstack([tfidf_nb.transform(X_test_df['statement']), X_test_cat])

nb_tfidf = ComplementNB(alpha=2.0, norm=False)
scores_tfidf = cross_val_score(nb_tfidf, X_train_tfidf_nb, y_train, cv=5, scoring='f1_macro')

print(f'TF-IDF + ComplementNB:      {scores_tfidf.mean():.4f} (+/-{scores_tfidf.std():.4f})')
print(f'CountVectorizer + ComplementNB: 0.6044')
print(f'Diferencia:                 {scores_tfidf.mean() - 0.6044:+.4f}')

TF-IDF + ComplementNB:      0.5951 (+/-0.0083)
CountVectorizer + ComplementNB: 0.6044
Diferencia:                 -0.0093


In [39]:
# Ampliar n-gramas a trigramas
vect_trigram = CountVectorizer(ngram_range=(1, 3), binary=True, min_df=5)
X_train_trigram = hstack([vect_trigram.fit_transform(X_train_df['statement']), X_train_cat])
X_test_trigram = hstack([vect_trigram.transform(X_test_df['statement']), X_test_cat])

nb_trigram = ComplementNB(alpha=2.0, norm=False)
scores_trigram = cross_val_score(nb_trigram, X_train_trigram, y_train, cv=5, scoring='f1_macro')

print(f'N-gramas (1,3): {scores_trigram.mean():.4f} (+/-{scores_trigram.std():.4f})')
print(f'N-gramas (1,2): 0.6044')
print(f'Diferencia:     {scores_trigram.mean() - 0.6044:+.4f}')

N-gramas (1,3): 0.6025 (+/-0.0046)
N-gramas (1,2): 0.6044
Diferencia:     -0.0019


In [40]:
from sklearn.naive_bayes import MultinomialNB, BernoulliNB

modelos_nb = {
    'ComplementNB (actual)': ComplementNB(alpha=2.0, norm=False),
    'MultinomialNB': MultinomialNB(alpha=2.0),
    'BernoulliNB': BernoulliNB(alpha=2.0)
}

print('Comparación variantes Naive Bayes:')
print()
for nombre, modelo in modelos_nb.items():
    scores = cross_val_score(modelo, X_train_combined, y_train, cv=5, scoring='f1_macro')
    print(f'{nombre:<30} {scores.mean():.4f} (+/-{scores.std():.4f})')

Comparación variantes Naive Bayes:

ComplementNB (actual)          0.6044 (+/-0.0046)
MultinomialNB                  0.6044 (+/-0.0052)
BernoulliNB                    0.5905 (+/-0.0138)


In [41]:
from scipy.sparse import csr_matrix

def text_features_nb(df_col):
    features = pd.DataFrame()
    features['length'] = df_col.str.len()
    features['word_count'] = df_col.str.split().str.len()
    features['avg_word_length'] = df_col.apply(lambda x: np.mean([len(w) for w in x.split()]) if len(x.split()) > 0 else 0)
    features['exclamation'] = df_col.str.count('!')
    features['question'] = df_col.str.count('\?')
    features['uppercase_ratio'] = df_col.apply(lambda x: sum(1 for c in x if c.isupper()) / len(x) if len(x) > 0 else 0)
    features['digit_count'] = df_col.str.count('\d')
    features['comma_count'] = df_col.str.count(',')
    return csr_matrix(features.values)

X_train_manual_nb = text_features_nb(train['statement'])
X_test_manual_nb = text_features_nb(X_test_df['statement'])

# Combinar con las features anteriores
X_train_manual_combined = hstack([X_train_combined, X_train_manual_nb])
X_test_manual_combined = hstack([X_test_combined, X_test_manual_nb])

nb_manual = ComplementNB(alpha=2.0, norm=False)
scores_manual = cross_val_score(nb_manual, X_train_manual_combined, y_train, cv=5, scoring='f1_macro')

print(f'Con features manuales:    {scores_manual.mean():.4f} (+/-{scores_manual.std():.4f})')
print(f'Sin features manuales:    0.6044')
print(f'Diferencia:               {scores_manual.mean() - 0.6044:+.4f}')

Con features manuales:    0.6017 (+/-0.0058)
Sin features manuales:    0.6044
Diferencia:               -0.0027


In [42]:
pipeline_v2 = Pipeline([
    ('vect', CountVectorizer(ngram_range=(1, 2))),
    ('clf', ComplementNB())
])

param_grid_v2 = {
    'vect__binary': [True],              # ya sabemos que True es mejor
    'vect__min_df': [3, 5, 7, 10],       # antes probamos 1,2,5 — ahora más valores
    'vect__max_features': [None, 20000, 50000],  # limitar vocabulario
    'vect__ngram_range': [(1,2), (1,3)], # incluir trigramas en el grid
    'clf__alpha': [1.5, 2.0, 2.5, 3.0], # explorar alrededor del óptimo 2.0
    'clf__norm': [False]                 # ya sabemos que False es mejor
}

grid_v2 = GridSearchCV(
    pipeline_v2,
    param_grid_v2,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)

grid_v2.fit(X_train, y_train)

print(f'Mejor F1-Macro: {grid_v2.best_score_:.4f}')
print()
print('Mejores parámetros:')
for param, valor in grid_v2.best_params_.items():
    print(f'  {param}: {valor}')

Fitting 5 folds for each of 96 candidates, totalling 480 fits
Mejor F1-Macro: 0.5988

Mejores parámetros:
  clf__alpha: 3.0
  clf__norm: False
  vect__binary: True
  vect__max_features: None
  vect__min_df: 7
  vect__ngram_range: (1, 2)


In [43]:
# Nuevos parámetros encontrados: alpha=3.0, min_df=7
vect_v2 = CountVectorizer(ngram_range=(1, 2), binary=True, min_df=7)
X_train_text_v2 = vect_v2.fit_transform(X_train_df['statement'])
X_test_text_v2 = vect_v2.transform(X_test_df['statement'])

X_train_combined_v2 = hstack([X_train_text_v2, X_train_cat])
X_test_combined_v2 = hstack([X_test_text_v2, X_test_cat])

nb_v2 = ComplementNB(alpha=3.0, norm=False)
scores_v2 = cross_val_score(nb_v2, X_train_combined_v2, y_train, cv=5, scoring='f1_macro')

print(f'alpha=3.0, min_df=7 + categóricas: {scores_v2.mean():.4f} (+/-{scores_v2.std():.4f})')
print(f'alpha=2.0, min_df=5 + categóricas: 0.6044')
print(f'Diferencia:                         {scores_v2.mean() - 0.6044:+.4f}')

alpha=3.0, min_df=7 + categóricas: 0.6084 (+/-0.0094)
alpha=2.0, min_df=5 + categóricas: 0.6044
Diferencia:                         +0.0040


In [44]:
resultados_fino = []

for alpha in [2.5, 3.0, 3.5, 4.0]:
    for min_df in [5, 7, 10]:
        vect_tmp = CountVectorizer(ngram_range=(1, 2), binary=True, min_df=min_df)
        X_tmp = hstack([vect_tmp.fit_transform(X_train_df['statement']), X_train_cat])
        nb_tmp = ComplementNB(alpha=alpha, norm=False)
        scores_tmp = cross_val_score(nb_tmp, X_tmp, y_train, cv=5, scoring='f1_macro')
        resultados_fino.append({
            'alpha': alpha,
            'min_df': min_df,
            'f1_mean': scores_tmp.mean(),
            'f1_std': scores_tmp.std()
        })
        print(f'alpha={alpha}, min_df={min_df}: {scores_tmp.mean():.4f} (+/-{scores_tmp.std():.4f})')

mejor = max(resultados_fino, key=lambda x: x['f1_mean'])
print()
print(f'Mejor combinación: alpha={mejor["alpha"]}, min_df={mejor["min_df"]} → F1={mejor["f1_mean"]:.4f}')

alpha=2.5, min_df=5: 0.6057 (+/-0.0062)
alpha=2.5, min_df=7: 0.6047 (+/-0.0083)
alpha=2.5, min_df=10: 0.6052 (+/-0.0054)
alpha=3.0, min_df=5: 0.6055 (+/-0.0046)
alpha=3.0, min_df=7: 0.6084 (+/-0.0094)
alpha=3.0, min_df=10: 0.6052 (+/-0.0056)
alpha=3.5, min_df=5: 0.6048 (+/-0.0049)
alpha=3.5, min_df=7: 0.6108 (+/-0.0103)
alpha=3.5, min_df=10: 0.6049 (+/-0.0033)
alpha=4.0, min_df=5: 0.6026 (+/-0.0072)
alpha=4.0, min_df=7: 0.6086 (+/-0.0085)
alpha=4.0, min_df=10: 0.6061 (+/-0.0039)

Mejor combinación: alpha=3.5, min_df=7 → F1=0.6108


In [45]:
resultados_fino2 = []

for alpha in [3.0, 3.2, 3.5, 3.8, 4.0]:
    for min_df in [6, 7, 8]:
        vect_tmp = CountVectorizer(ngram_range=(1, 2), binary=True, min_df=min_df)
        X_tmp = hstack([vect_tmp.fit_transform(X_train_df['statement']), X_train_cat])
        nb_tmp = ComplementNB(alpha=alpha, norm=False)
        scores_tmp = cross_val_score(nb_tmp, X_tmp, y_train, cv=5, scoring='f1_macro')
        resultados_fino2.append({
            'alpha': alpha,
            'min_df': min_df,
            'f1_mean': scores_tmp.mean(),
            'f1_std': scores_tmp.std()
        })
        print(f'alpha={alpha}, min_df={min_df}: {scores_tmp.mean():.4f} (+/-{scores_tmp.std():.4f})')

mejor2 = max(resultados_fino2, key=lambda x: x['f1_mean'])
print()
print(f'Mejor combinación: alpha={mejor2["alpha"]}, min_df={mejor2["min_df"]} → F1={mejor2["f1_mean"]:.4f}')

alpha=3.0, min_df=6: 0.6068 (+/-0.0070)
alpha=3.0, min_df=7: 0.6084 (+/-0.0094)
alpha=3.0, min_df=8: 0.6041 (+/-0.0080)
alpha=3.2, min_df=6: 0.6069 (+/-0.0074)
alpha=3.2, min_df=7: 0.6099 (+/-0.0095)
alpha=3.2, min_df=8: 0.6040 (+/-0.0076)
alpha=3.5, min_df=6: 0.6069 (+/-0.0083)
alpha=3.5, min_df=7: 0.6108 (+/-0.0103)
alpha=3.5, min_df=8: 0.6060 (+/-0.0078)
alpha=3.8, min_df=6: 0.6071 (+/-0.0061)
alpha=3.8, min_df=7: 0.6094 (+/-0.0086)
alpha=3.8, min_df=8: 0.6066 (+/-0.0088)
alpha=4.0, min_df=6: 0.6077 (+/-0.0077)
alpha=4.0, min_df=7: 0.6086 (+/-0.0085)
alpha=4.0, min_df=8: 0.6074 (+/-0.0072)

Mejor combinación: alpha=3.5, min_df=7 → F1=0.6108


In [46]:
vect_best = CountVectorizer(ngram_range=(1, 2), binary=True, min_df=7)
X_train_best = hstack([vect_best.fit_transform(X_train_df['statement']), X_train_cat])
X_test_best = hstack([vect_best.transform(X_test_df['statement']), X_test_cat])

nb_best = ComplementNB(alpha=3.5, norm=False)
y_proba_best = cross_val_predict(nb_best, X_train_best, y_train, cv=5, method='predict_proba')

print('Threshold tuning alpha=3.5, min_df=7:')
print()
for threshold in [0.35, 0.40, 0.45, 0.50, 0.55]:
    y_pred_thresh = (y_proba_best[:, 1] >= threshold).astype(int)
    f1 = f1_score(y_train, y_pred_thresh, average='macro')
    f1_fake = f1_score(y_train, y_pred_thresh, average=None)[0]
    f1_real = f1_score(y_train, y_pred_thresh, average=None)[1]
    print(f'Threshold {threshold:.2f}: F1-Macro={f1:.4f} | F1-fake={f1_fake:.4f} | F1-real={f1_real:.4f}')

Threshold tuning alpha=3.5, min_df=7:

Threshold 0.35: F1-Macro=0.6053 | F1-fake=0.4700 | F1-real=0.7405
Threshold 0.40: F1-Macro=0.6070 | F1-fake=0.4844 | F1-real=0.7296
Threshold 0.45: F1-Macro=0.6068 | F1-fake=0.4949 | F1-real=0.7187
Threshold 0.50: F1-Macro=0.6108 | F1-fake=0.5108 | F1-real=0.7108
Threshold 0.55: F1-Macro=0.6067 | F1-fake=0.5184 | F1-real=0.6949


In [48]:
# Entrenar con todos los datos de train
nb_best.fit(X_train_best, y_train)

# Predecir en test
y_pred_test = nb_best.predict(X_test_best)

# Crear submission
submission = pd.DataFrame({
    'id': test['id'],
    'label': y_pred_test
})

submission.to_csv('submission_naive_bayes.csv', index=False)
print('Submission actualizada como submission_naive_bayes.csv')
print()
print('Distribución de predicciones:')
print(submission['label'].value_counts())
print()
print('Resumen final del modelo:')
print(f'  Modelo:        Complement Naive Bayes')
print(f'  Features:      statement (CountVectorizer bigrams) + speaker + party + subject')
print(f'  Parámetros:    alpha=3.5, norm=False, binary=True, min_df=7')
print(f'  F1-Macro CV:   0.6108')
print(f'  Threshold:     0.50')
print()
print('Evolución del modelo:')
print(f'  Solo statement (grid search inicial): 0.5944')
print(f'  + features categóricas:               0.6044')
print(f'  + alpha=3.5, min_df=7:                0.6108')

Submission actualizada como submission_naive_bayes.csv

Distribución de predicciones:
label
1    2302
0    1534
Name: count, dtype: int64

Resumen final del modelo:
  Modelo:        Complement Naive Bayes
  Features:      statement (CountVectorizer bigrams) + speaker + party + subject
  Parámetros:    alpha=3.5, norm=False, binary=True, min_df=7
  F1-Macro CV:   0.6108
  Threshold:     0.50

Evolución del modelo:
  Solo statement (grid search inicial): 0.5944
  + features categóricas:               0.6044
  + alpha=3.5, min_df=7:                0.6108
